In [2]:
!pip install tqdm bs4 requests fake_useragent free-proxy pandas musicbrainzngs

# 1. Creating the dataset for the project

In [2]:
from tqdm import tqdm
from bs4 import BeautifulSoup
import requests
from fake_useragent import UserAgent
from fp.fp import FreeProxy
import re
import time
import random
import os
import pandas as pd
import musicbrainzngs

In [ ]:
ua = UserAgent()

headers = {"User-Agent": 'Mozilla/5.0 (iPhone; CPU iPhone OS 18_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/18.2 Mobile/15E148 Safari/604.1'}
proxy_url = FreeProxy(timeout=1,
                      rand=True).get()

print(f"Using working proxy: {proxy_url}")
proxies = {
    "http": proxy_url,
    "https": proxy_url,
}

response = requests.get('https://www.azlyrics.com/t/taylorswift.html', 
                        headers=headers, proxies=proxies)
path_page = BeautifulSoup(response.text, 'html.parser')

In [ ]:
errors = []
not_needed = []
main_path = 'https://www.azlyrics.com'
all_songs = path_page.find_all('div', class_='listalbum-item')

for song_i in tqdm(all_songs):
    if 'azlyrics' not in song_i.a['href']:
        song_link = main_path + song_i.a['href']
    else:
        song_link = song_i.a['href']
    
    song_name = song_i.text

    if f'{song_name}.txt' in os.listdir('.lyrics/'):
        continue

    if "Taylor's Version" in song_name:
        if "(From The Vault)" not in song_name:
            not_needed.append(song_name)
            continue
    if 'Remix' in song_name:
        not_needed.append(song_name)
        continue
    if 'Sad Girl Autumn Version' in song_name:
        not_needed.append(song_name)
        continue
    if 'Music-Video Version' in song_name:
        not_needed.append(song_name)
        continue
    if 'Music Video Version' in song_name:
        not_needed.append(song_name)
        continue
    if 'Acoustic' in song_name:
        not_needed.append(song_name)
        continue
    if 'Demo' in song_name:
        not_needed.append(song_name)
        continue
    if 'Piano Version' in song_name:
        not_needed.append(song_name)
        continue
    if 'Live from' in song_name:
        not_needed.append(song_name)
        continue
    if 'Tour Live' in song_name:
        not_needed.append(song_name)
        continue
    if 'World Tour' in song_name:
        not_needed.append(song_name)
        continue
    if 'A Message From Taylor' in song_name:
        not_needed.append(song_name)
        continue
    if 'Meet Me At Midnight' in song_name:
        not_needed.append(song_name)
        continue

    try:
        song_data = requests.get(song_link, headers=headers, proxies=proxies)
        song_page = BeautifulSoup(song_data.text, 'html.parser')
        with open(f'.lyrics/{song_name}.txt', 'w') as f:
            print(song_page, file=f)
        time.sleep(random.uniform(1.5, 5))
    except:
        errors.append(song_link)
print(errors)

  0%|          | 0/487 [00:00<?, ?it/s]

100%|██████████| 487/487 [01:29<00:00,  5.44it/s]

[]


In [ ]:
print(not_needed[:10])

['Forever & Always (Piano Version)(Platinum Edition)', "Fearless (Taylor's Version)", "Fifteen (Taylor's Version)", "Love Story (Taylor's Version)", "Hey Stephen (Taylor's Version)", "White Horse (Taylor's Version)", "You Belong With Me (Taylor's Version)", "Breathe (Taylor's Version)", "Tell Me Why (Taylor's Version)", "You're Not Sorry (Taylor's Version)"]


In [3]:
with open('pen_categories/fountain pen.txt') as f:
    fountain_pen = list(map(lambda x: x.strip(), 
                            f.readlines()))
with open('pen_categories/glitter gel pen.txt') as f:
    glitter_pen = list(map(lambda x: x.strip(), 
                            f.readlines()))
with open('pen_categories/quill pen.txt') as f:
    quill_pen = list(map(lambda x: x.strip(), 
                            f.readlines()))

path_list = []
artist_name_list = []
song_name_list = []
song_writers_list = []
song_album_list = []
song_year_list = []
song_lyrics_list = []
song_category_list = []
    
for song_i in tqdm(os.listdir('.lyrics/')):
    with open(f'.lyrics/{song_i}') as f:
        path_list.append(song_i)
        txt_page = BeautifulSoup(f.read(), 'html.parser')
        try:
            artist_name = re.search('(?<=ArtistNameNc = ").*(?=";)',
                                txt_page.find_all('script')[1].text).group()
        except:
            artist_name = pd.NA
        try:
            song_name = re.search('(?<=SongNameNc = ").*(?=";)',
                                txt_page.find_all('script')[1].text).group()
            if song_name in fountain_pen:
                song_category = 'Fountain Pen'
            elif song_name in glitter_pen:
                song_category = 'Glitter Gel Pen'
            elif song_name in quill_pen:
                song_category = 'Quill Pen'
            else:
                song_category = pd.NA
        except:
            song_name = pd.NA
            song_category = pd.NA
        try:
            song_writers = re.search('(?<=Writer\\(s\\):).*',
                                     txt_page.text).group().strip().split(',')
        except:
            song_writers = pd.NA
        try:
            song_album = txt_page.find_all('div', class_='songinalbum_title')[0].text
            if 'You May Also Like' not in song_album:
                song_year = int(re.search('(?<=\\()[0-9]+',
                                    song_album).group())
                song_album = re.search('(?<=").*(?=")',
                                    song_album).group()
            else:
                song_year = pd.NA
                song_album = pd.NA
        except:
            song_album = pd.NA
            song_year = pd.NA
        
        try:
            song_lyrics = txt_page.text.split('\n\n\n\n')[28].replace('\n\n', '\n')
        except:
            song_lyrics = pd.NA
        

        artist_name_list.append(artist_name)
        song_name_list.append(song_name)
        song_writers_list.append(song_writers)
        song_album_list.append(song_album)
        song_year_list.append(song_year)
        song_lyrics_list.append(song_lyrics)
        song_category_list.append(song_category)

dataset = pd.DataFrame(
    {
        'Artist': artist_name_list,
        'Song Name': song_name_list,
        'Writers': song_writers_list,
        'Album': song_album_list,
        'Release Year': song_year_list,
        'Lyrics': song_lyrics_list,
        'Pen Category': song_category_list,
        'Path': path_list
    }
)
     
        

  0%|          | 0/373 [00:00<?, ?it/s]

100%|██████████| 373/373 [00:06<00:00, 60.00it/s]


In [4]:
print(f'The size of the dataset is {dataset.shape[0]}',
      'Among this:',
      f'{dataset[dataset['Song Name'].notna()].shape[0]} entries include the song name',
      f'{dataset[dataset['Lyrics'].notna()].shape[0]} entries include the song lyrics',
      f'{dataset[dataset['Writers'].notna()].shape[0]} entries include the song name',
      f'{dataset[dataset['Album'].notna()].shape[0]} entries include the album name',
      f'{dataset[dataset['Release Year'].notna()].shape[0]} entries include the release year',
      f'{dataset[dataset['Pen Category'].notna()].shape[0]} entries include the pen category name',
      sep = '\n')
dataset[dataset.Album != 'Sounds Of The Season: The Taylor Swift Holiday Collection']
dataset.to_csv('.data/dataset.csv', index=False)

The size of the dataset is 373
Among this:
373 entries include the song name
373 entries include the song lyrics
272 entries include the song name
269 entries include the album name
269 entries include the release year
55 entries include the pen category name


In [5]:
dataset

,Artist,Song Name,Writers,Album,Release Year,Lyrics,Pen Category,Path
0,Taylor Swift,Cassandra,"[Aaron Brooking Dessner, Taylor Alison Swift]",THE TORTURED POETS DEPARTMENT,2024,I was in my new house placing daydreams\nPatch...,NaN,Cassandra(THE ANTHOLOGY Edition Bonus Track).txt
1,Taylor Swift,ivy,"[Jack Michael Antonoff, Aaron Brooking Dessne...",evermore,2020,How's one to know?\nI'd meet you where the spi...,Quill Pen,ivy.txt
2,Taylor Swift,You Are In Love,"[Taylor Swift, Jack Michael Antonoff]",1989,2014,"One look, dark room\nMeant just for you\nTime ...",NaN,You Are In Love(Deluxe Edition Bonus Track).txt
3,Taylor Swift,Bad Blood,"[Shellback, Taylor Swift, Martin Max]",1989,2014,"'Cause, baby, now we got bad blood\nYou know i...",NaN,Bad Blood.txt
4,Taylor Swift,"Would've, Could've, Should've","[Taylor Swift, Aaron Brooking Dessner]",Midnights,2022,"If you would've blinked, then I would've\nLook...",NaN,"Would've, Could've, Should've(3am Edition Bonu..."
...,...,...,...,...,...,...,...,...
368,Taylor Swift,All Of The Girls You Loved Before,"[Adam King Feeney, Taylor Swift, Matthew Ada...",NaN,<NA>,When you think of all the late nights\nLame fi...,NaN,All Of The Girls You Loved Before.txt
369,Taylor Swift,Mean,[Taylor Swift],Speak Now,2010,"You, with your words like knives\nAnd swords a...",NaN,Mean.txt
370,Taylor Swift,Carolina,<NA>,NaN,<NA>,O Carolina creeks\nRunning through my veins\nL...,Quill Pen,"Carolina(from ""Where The Crawdads Sing"" soundt..."
371,Taylor Swift,Who's Afraid Of Little Old Me?,<NA>,THE TORTURED POETS DEPARTMENT,2024,"The ""Who's Who"" of ""Who's That?""\nIs poised fo...",NaN,Who's Afraid Of Little Old Me?.txt


In [3]:
musicbrainzngs.set_useragent("Example music app", "0.1", "http://example.com/music")

In [8]:
song_title = 'Carolina'
artist = 'Taylor Swift'
query = f'recording:"{song_title}" AND artist:"{artist}"'
result = musicbrainzngs.search_recordings(query=query, limit=5)
for index, rec in enumerate(result.get('recording-list', [])):
    print(f"\n[{index + 1}] Title: {rec['title']}")
    print(f"    Artist: {rec.get('artist-credit-phrase', 'Unknown')}")
    print(f"    ID: {rec['id']}")

KeyboardInterrupt: 

In [212]:
result = musicbrainzngs.get_artist_by_id('20244d07-534f-4eff-b4d4-930878889970',
              includes=["release-groups"], release_type=["album", "ep", "single"])
for release_group in result["artist"]["release-group-list"]:
    print("{title} ({type})".format(title=release_group["title"],
                                    type=release_group["type"]))

Taylor Swift (Album)
Fearless (Album)
Speak Now (Album)
Red (Album)
1989 (Album)
reputation (Album)
Lover (Album)
folklore (Album)
evermore (Album)
Fearless (Taylor’s version) (Album)
Red (Taylor’s version) (Album)
Midnights (Album)
Speak Now (Taylor’s version) (Album)
1989 (Taylor’s version) (Album)
THE TORTURED POETS DEPARTMENT (Album)
The Life of a Showgirl (Album)
Complete Album Collection (Compilation)
reputation Stadium Tour Surprise Song Playlist (Compilation)
Speak Now: World Tour Live (Live)
Live From Clear Channel Stripped 2008 (Live)
Lover (live from Paris) (Live)
folklore: the long pond studio sessions (Live)
Taylor Swift Karaoke: Taylor Swift (Remix)
Taylor Swift Karaoke: Fearless (Remix)
Taylor Swift Karaoke: Speak Now (Remix)


In [213]:
help(musicbrainzngs.get_artist_by_id)

Help on function get_artist_by_id in module musicbrainzngs.musicbrainz:

get_artist_by_id(id, includes=[], release_status=[], release_type=[])
    Get the artist with the MusicBrainz `id` as a dict with an 'artist' key.

    *Available includes*: recordings, releases, release-groups, works, various-artists, discids, media, isrcs, aliases, annotation, area-rels, artist-rels, label-rels, place-rels, event-rels, recording-rels, release-rels, release-group-rels, series-rels, url-rels, work-rels, instrument-rels, tags, user-tags, ratings, user-ratings

